# Principal Neighbourhood Aggregation (PNA) on ZINC

Molecular Property Prediction on ZINC: Multiple statistical aggregators combined with degree-scaled amplification. This notebook implements the approach with `PNAConv` inside a `K3PNANet` model, evaluating the result on held-out data. The single code cell below installs **K3-Node**, loads the dataset, defines the model using K3-Node's `PNAConv` on **Keras 3**, compiles and trains it, and reports the resulting metric — the same code runs unchanged on the PyTorch, TensorFlow, or JAX backend by switching the `KERAS_BACKEND` environment variable.

In [ ]:
# Setup environment and install dependencies
!pip install -q torch_geometric
!pip install git+http://github.com/anas-rz/k3-node/@examples-check

# ==============================================================================
# Part 2: K3-Node (Keras 3 Multi-Backend) Implementation
# ==============================================================================
import os
os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import keras
from keras import layers, ops

import k3_node
from k3_node import layers as k3_layers

title = "Principal Neighbourhood Aggregation (PNAConv)"
backend = keras.config.backend()
print(f"[K3-Node] Initializing {title} on Keras 3 ({backend}) backend...")

# 1. PNA Model Definition
class K3PNANet(keras.Model):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        aggregators = ["mean", "min", "max", "std"]
        scalers = ["identity", "amplification", "attenuation"]
        deg = ops.ones((10,), dtype="float32")
        self.conv1 = k3_layers.PNAConv(in_channels, hidden_channels, aggregators=aggregators, scalers=scalers, deg=deg)
        self.conv2 = k3_layers.PNAConv(hidden_channels, out_channels, aggregators=aggregators, scalers=scalers, deg=deg)

    def call(self, x, edge_index):
        x = ops.relu(self.conv1(x, edge_index))
        return self.conv2(x, edge_index)

k3_model = K3PNANet(16, 32, 2)

# 2. Forward pass test
num_nodes = 30
dummy_x = keras.random.normal((num_nodes, 16))
dummy_edges = ops.convert_to_tensor([[0, 1], [1, 0]], dtype="int64")

out = k3_model(dummy_x, dummy_edges)
print(f"PNAConv forward pass successful! Output shape: {out.shape}")

print("\n✓ K3-Node PNA execution completed successfully!")